# Ingestão — Landing Zone

Busca a cotação atual dos 4 tickers do projeto (PETR4, VALE3, MGLU3, ITUB4) na API `brapi.dev`
e grava a resposta bruta (JSON) no Volume `poc_b3_modernizacao.landing.raw`, sem nenhum parsing
ou tratamento — cópia fiel da fonte, conforme ADR-01.

Esta ingestão é independente do workflow KNIME: ambos consomem a mesma API de forma separada,
permitindo a reconciliação posterior entre as duas implementações.

**Entrada:** API `brapi.dev`
**Saída:** arquivo JSON bruto em `/Volumes/poc_b3_modernizacao/landing/raw/data=AAAA-MM-DD/`

In [0]:
%run ../setup/01_utilitarios_pipeline

In [0]:
# observabilidade - marca inicio da execucao
from datetime import datetime
inicio_execucao = datetime.now()

In [0]:
# imports 

import requests
import json
from datetime import datetime

In [0]:
# widgets - parametros de execucao
dbutils.widgets.text("data_referencia", "", "Data de referencia (AAAA-MM-DD)")
dbutils.widgets.dropdown("ticker", "todos", ["todos", "PETR4", "VALE3", "MGLU3", "ITUB4"], "Ticker")
dbutils.widgets.dropdown("modo_execucao", "agendado", ["agendado", "reprocessamento_manual"], "Modo de execucao")

In [0]:
# configuracao - le os widgets
_ticker_selecionado = dbutils.widgets.get("ticker")
TICKERS = ["PETR4", "VALE3", "MGLU3", "ITUB4"] if _ticker_selecionado == "todos" else [_ticker_selecionado]
DATA_REFERENCIA = dbutils.widgets.get("data_referencia").strip() or datetime.now().strftime("%Y-%m-%d")
MODO_EXECUCAO = dbutils.widgets.get("modo_execucao")
VOLUME_PATH = f"/Volumes/poc_b3_modernizacao/landing/raw/data={DATA_REFERENCIA}"

print(f"Tickers: {TICKERS}")
print(f"Data de referencia: {DATA_REFERENCIA}")
print(f"Modo de execucao: {MODO_EXECUCAO}")
print(f"Caminho do volume: {VOLUME_PATH}")

In [0]:
# execucao principal - busca api, grava volume, valida, registra observabilidade (com tratamento de erro)
try:
    resultados = []
    for ticker in TICKERS:
        url = f"https://brapi.dev/api/quote/{ticker}"
        response = requests.get(url)
        response.raise_for_status()
        resultados.append({
            "ticker": ticker,
            "status_code": response.status_code,
            "body": response.json()
        })
        print(f"{ticker}: status {response.status_code}")

    dbutils.fs.mkdirs(VOLUME_PATH)
    nome_arquivo = f"{VOLUME_PATH}/cotacoes_{DATA_REFERENCIA}.json"
    with open(nome_arquivo, "w") as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)
    print(f"Arquivo gravado em: {nome_arquivo}")

    arquivos = dbutils.fs.ls(VOLUME_PATH)
    print(f"Arquivos em {VOLUME_PATH}:")
    for arquivo in arquivos:
        print(f"  {arquivo.name} ({arquivo.size} bytes)")

    registrar_execucao(
        notebook="01_ingestao_landing",
        data_referencia=DATA_REFERENCIA,
        modo_execucao=MODO_EXECUCAO,
        status="sucesso",
        inicio=inicio_execucao,
        fim=datetime.now(),
    )

except Exception as e:
    registrar_execucao(
        notebook="01_ingestao_landing",
        data_referencia=DATA_REFERENCIA,
        modo_execucao=MODO_EXECUCAO,
        status="falha",
        inicio=inicio_execucao,
        fim=datetime.now(),
        mensagem_erro=str(e),
    )
    raise

### Visualização

In [0]:
# valida registro de observabilidade
display(spark.table("poc_b3_modernizacao.observability.pipeline_runs"))